# Wskazania a rozrzut temperatur nóg

Pomysł użytkownika (2026-11-05): wskazania zmieniają się nie od wspólnej temperatury, tylko od tego, jak bardzo temperatury nóg odstają od swojej średniej. Miara: **rozrzut** = odchylenie standardowe 4 czujników w danej chwili (0, gdy wszystkie nogi mają tę samą temperaturę). Oczekiwanie: rośnie rozrzut → spada wskazanie, rozrzut wraca do zera → wskazanie wraca do poziomu odniesienia.

Zbiór: baza `2026-11-02 obiekt`, sesja 32, wariant `CURRENT_VARIANT`; tylko fragmenty bez zdarzeń (margines 30 s) bez zakłóceń, z co najmniej 60 minutami; mediana wskazań i średnie temperatur w minutach. Obliczenia w `dms_analysis.temperature_spread`.

**Dopasowanie** liczone wewnątrz fragmentów: od wskazań i cech odejmowana jest średnia fragmentu (masa materiału we fragmencie się nie zmienia, ale między fragmentami tak), potem prosta regresja. Model „szybkość zmian średniej” = zmiana średniej temperatury w °C/h (różnica ±30 min) — sprawdzenie hipotezy 2 z `zakres.md` §1C obok rozrzutu.

**Uwaga:** praktycznie cały wynik pochodzi z jednego fragmentu (30.10 wieczór – 2.11 rano, weekend), czyli z 2,5 doby. Kolejne minuty nie są niezależne, więc R² i korelacje są opisowe, bez testów istotności.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import display

from dms_analysis.export import PREPARED_DB
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.temperature_spread import (
    INCREMENT_STEPS,
    change_summary,
    comparison,
    correction_figure,
    correction_summary,
    day_coefficients,
    figure,
    fit,
    fragment_summary,
    increment_scatter,
    increment_summary,
    method_summary,
    minute_table,
    scatter,
    semi_synthetic,
)

SOURCE = "sqlite-2026-11-02"
SESSION_ID = 32

session = load_session(PREPARED_DB, SOURCE, SESSION_ID, CURRENT_VARIANT)
table = minute_table(session)
display(fragment_summary(table))

## Porównanie modeli

Współczynniki są **wyliczone** metodą najmniejszych kwadratów (nie przyjęte z góry): dla każdego modelu wybierane są takie liczby, przy których suma kwadratów różnic między wskazaniami a przewidywaniem jest najmniejsza. Jednostki: `spread` w kg na 1 °C rozrzutu, `mean` w kg na 1 °C średniej temperatury, `rate` w kg na 1 °C/h szybkości zmian średniej, `spread_load` w % poziomu fragmentu na 1 °C rozrzutu.

**Wersja procentowa** (`rozrzut procentowo od poziomu fragmentu`): zmiana wskazania = współczynnik × rozrzut × poziom fragmentu / 100, gdzie poziom fragmentu = średnie wskazanie we fragmencie (przybliżenie obciążenia silosu). Pasuje do mechanizmu zależnego od obciążenia (np. przenoszenie ciężaru między tensometrami); wersja w kg pasuje do sił niezależnych od obciążenia (rozszerzalność konstrukcji, sztywne połączenia). **Na jednym poziomie zapełnienia obie wersje dają to samo** (−3,03 % × 47 t ≈ −1422 kg); rozstrzygnąć można dopiero na fragmentach przy różnym zapełnieniu.

Odchylenie std wskazań i reszt liczone wewnątrz fragmentów. R² = jaka część zmienności wskazań we fragmencie jest wyjaśniona.

In [ ]:
display(comparison(table))

## Wskazania i dopasowanie samym rozrzutem

Od góry: wskazania (czarne) z dopasowaniem (czerwone), reszta, rozrzut, średnia temperatura. Jeśli pomysł jest trafny, czerwona linia powinna iść za czarną, a reszta nie powinna mieć wyraźnego rytmu dobowego.

In [ ]:
spread_fit = fit(table, ["spread"])
figure(spread_fit, "dopasowanie: rozrzut").show()

In [ ]:
scatter(spread_fit).show()

## Rozrzut i szybkość zmian średniej temperatury

Ten sam wykres dla modelu z dwiema cechami — czy reszta po samym rozrzucie (ujemna rano przy nagrzewaniu, dodatnia po południu przy stygnięciu) to efekt szybkości zmian temperatury.

In [ ]:
rate_fit = fit(table, ["spread", "rate"])
figure(rate_fit, "dopasowanie: rozrzut + szybkość zmian średniej").show()

## Wersja w przyrostach: zmiana wskazania wobec zmiany rozrzutu

Bez poziomu odniesienia: dla par minut odległych o 1 albo 10 minut (w tym samym fragmencie) zmiana wskazania porównana ze zmianą rozrzutu. Prosta przechodzi przez zero (brak zmiany rozrzutu → brak zmiany wskazania). Jeśli związek jest natychmiastowy, nachylenie powinno być podobne jak w wersji z poziomami; przyrosty są mniej podatne na korelację pozorną, ale bardziej zaszumione (szczególnie co 1 min, gdzie zmiany rozrzutu są rzędu rozdzielczości czujników 0,0625 °C).

In [ ]:
display(increment_summary(table))

In [ ]:
for steps in INCREMENT_STEPS:
    increment_scatter(table, steps).show()

## Testy półsyntetyczne: czy poprawka rozrzutem zmniejsza błąd odsypu

Jak w `synthetic.ipynb`: w losowe miejsca fragmentów bez zdarzeń sesji 32 wstawiany jest sztuczny odsyp (rampa o masie i czasie trwania losowanym z prawdziwych odsypów tej sesji, zaokrąglenie do 20 kg), 2000 prób, potem błąd wyznaczonej masy = wynik − prawdziwa masa. Te same próby liczone na trzech wersjach wskazań:

- **bez poprawki**,
- **poprawka, nachylenie z całej sesji**: wskazanie − nachylenie × rozrzut, nachylenie z całej sesji (−1422 kg/°C) — wynik optymistyczny, bo nachylenie dopasowane na tych samych danych, na których jest test,
- **poprawka, nachylenie z pozostałych dni** (walidacja krzyżowa po dniach): dla próby z danego dnia nachylenie liczone bez tego dnia — uczciwsze sprawdzenie, czy zależność przenosi się na nowy dzień.

Dwa modele poprawki, każdy w obu wersjach współczynników: **rozrzut** (wskazanie − a × rozrzut) i **rozrzut + szybkość zmian** (wskazanie − a × rozrzut − b × szybkość zmian średniej temperatury, model z R² 0,79). Szybkość zmian jest liczona tak jak przy dopasowaniu: różnica średnich minutowych 30 min po i 30 min przed, w °C/h — używa przyszłości, więc w tej postaci tylko offline. Zmienia się w skali godzin, więc w czasie 30-sekundowego odsypu jej wkład do różnicy przed–po jest bardzo mały; model różni się od samego rozrzutu głównie innym współczynnikiem rozrzutu.

Rozrzut do poprawki jest liczony co sekundę i wygładzony średnią z ostatnich 30 s (tylko przeszłość, więc da się to robić na bieżąco). Bez wygładzenia poprawka dodaje skoki rozdzielczości czujników (0,0625 °C → 20–40 kg) i w dzień zysk był mniejszy (std 41 zamiast 32 kg), a w nocy strata większa (17 zamiast 15 kg); okna 10–300 s porównane 2026-11-05, 30 s najlepsze w dzień.

Pory dnia według prawdziwego zegara (sesja 32 ma prawdziwy czas, inaczej niż sesja 9 w `synthetic.ipynb`).

In [ ]:
display(day_coefficients(table))

In [ ]:
results = semi_synthetic(session, table)
display(correction_summary(results))

### Błąd zależnie od tego, jak bardzo zmienił się rozrzut w czasie odsypu

Zmiana rozrzutu między 3 s przed początkiem a 3 s po końcu sztucznego odsypu (bez wygładzenia). Poprawka może pomóc tylko tam, gdzie rozrzut się zmienia.

In [ ]:
display(change_summary(results))

In [ ]:
correction_figure(results).show()

### Wszystkie metody wyznaczania masy (odchylenie std błędu, wszystkie pory dnia)

In [ ]:
display(method_summary(results))